In [1]:
import re
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
from collections import defaultdict

import sys
import os

# Get the absolute path to the target file's directory
scraper_dir = os.path.abspath('../../../scripts/data_collection')
sys.path.append(scraper_dir)

# Now import from the file (without the .py extension)
import fbref_scraper as fs






/Users/tylermartins/Documents/Real Projects/soccer-betting/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


# Shots on Target Data Scraper

### Historical Players Stats
Data Source: FBRef

In [2]:
# INPUTS
headers = {"User-Agent": "Mozilla/5.0"}
data_path = '../../data/raw'
schedule_url = "https://fbref.com/en/comps/9/schedule/Premier-League-Scores-and-Fixtures"
season = '2024-2025'
league = 'PL'

# If True, ignore current data and do full webscrape. Else, only scrape URLs where home_score is NaN
full_rerun = False

schedule_url = 'https://fbref.com/en/comps/9/2023-2024/schedule/2023-2024-Premier-League-Scores-and-Fixtures'


test_url = 'https://fbref.com/en/matches/a24b7a43/Manchester-City-Ipswich-Town-August-24-2024-Premier-League'


Schema:
| Column                              | Type  | Description                                  |
| ----------------------------------- | ----- | -------------------------------------------- |
| `shots_per90`                       | float | Total shots per 90                           |
| `shots_team_share`                  | float | Total shots per 90                           |
| `sot_per90`                         | float | Shots on target per 90                       |
| `sot_team_share`                    | float | Shots on target per 90 / Team SOT            |
| `xg_per90`                          | float | Expected goals per 90                        |
| `npxg_per90`                        | float | Non-penalty expected goals per 90            |
| `xg_team_share`                     | float | Expected goals per 90 / Team XG              |
| `xa_per90`                          | float | Expected assists per 90                      |
| `sca_per90`                         | float | Shot-creating actions per 90                 |
| `gca_per90`                         | float | Goal-creating actions per 90                 |
| `touches_att_third_per90`           | float | Touches in the attacking third per 90        |
| `touches_att_pen_area_per90`        | float | Touches in the penalty area per 90           |
| `miscontrols_per90`                 | float | Miscontrols per 90                           |
| `dispossessed_per90`                | float | Dispossessions per 90                        |
| `passes_received_per90`             | float | Passes received per 90                       |
| `progressive_passes_received_per90` | float | Progressive passes received per 90           |
| `take_ons_won_per90`                | float | Successful take-ons per 90                   |
| `take_ons_tackled_per90`            | float | Unsuccessful take-ons per 90                 |
| `recent_form_zscore`                | float | Z-score of recent SoT form vs season average |




Raw Data Schema:
| Column                        | Type         | Description                                                |
| ----------------------------- | ------------ | ---------------------------------------------------------- |
| `player_id`                   | string       | Unique player identifier                                   |
| `player_name`                 | string       | Full name of the player                                    |
| `team`                        | string       | Team name                                                  |
| `position`                    | string       | Player's position                                          |
| `age`                         | string       | Player's age (format: YY-DDD)                              |
| `minutes_played`              | float        | Total minutes played in the sample period                  |
| `shots_total`                 | int          | Total shots attempted (`shots`)                            |
| `shots_on_target`             | int          | Shots on target (`shots_on_target`)                        |
| `xg_total`                    | float        | Total expected goals (`xg`)                                |
| `npxg_total`                  | float        | Total non-penalty expected goals (`npxg`)                  |
| `xa_total`                    | float        | Total expected assists (`xg_assist`)                       |
| `sca_total`                   | int          | Total shot-creating actions (`sca`)                        |
| `gca_total`                   | int          | Total goal-creating actions (`gca`)                        |
| `touches_att_third`           | int          | Touches in attacking third (`touches_att_3rd`)             |
| `touches_att_pen_area`        | int          | Touches in attacking penalty area (`touches_att_pen_area`) |
| `miscontrols`                 | int          | Number of times player miscontrolled the ball              |
| `dispossessed`                | int          | Number of times player lost possession                     |
| `passes_received`             | int          | Total passes received (`rec`)                              |
| `progressive_passes_received` | int          | Progressive passes received (`prgR`)                       |
| `take_ons_won`                | int          | Total take-ons (`take_ons`)                                |
| `take_ons_won`                | int          | Successful take-ons (`take_ons_won`)                       |
| `take_ons_tackled`            | float        | Unsuccessful take-ons (`take_ons_tackled`)                 |
| `sot_game_log`                | list\[float] | SoT per game, for computing recent form z-score            |
| `team_shots`                  | float        | Team's total shots                                         |
| `team_shots_on_target`        | float        | Team's total shots on target                               |
| `team_xg`                     | float        | Team's total Expected Goals                                |






In [3]:
# ----------------------------------------------------
# Step 2: Get Shots on Target data from the Match Report page
# ----------------------------------------------------

def get_sot_stats(match_url):

    response = fs.fetch_url_with_backoff(match_url)


    soup = BeautifulSoup(response.text, 'html.parser')

    # Dictionary to hold merged stats for each player
    merged_player_stats = defaultdict(dict)

    # Get the Match Date
    date_div = soup.find('div', class_ = 'scorebox_meta')

    date_div_tag = date_div.find('a')
    if date_div_tag:
        date = date_div_tag.text.strip()
    else:
        date = date_div.text.strip()



    # Step 1: Find all player_stats divs by id
    player_stats_divs = soup.find_all('div', id=re.compile(r'all_player_stats'))





    table_fields_dict = {
        'summary': [
            'player',
            'position',
            'team',
            'age',
            'minutes',
            'goals',
            'shots',
            'shots_on_target',
            'xg',
            'npxg',
            'xg_assist',
            'sca',
            'gca'
        ],
        'possession': [
            'miscontrols',
            'dispossessed',
            'passes_received',
            'take_ons',
            'take_ons_won',
            'take_ons_tackled',
            'touches_att_3rd',
            'touches_att_pen_area'
        ]
    }

    for section, fields in table_fields_dict.items():
        for div in player_stats_divs:
            header = div.find('h2')
            if not header:
                continue
            raw_team_name = header.text.strip()
            team_name = raw_team_name.replace(' Player Stats', '')

            tables = div.find_all('table', id=re.compile(section))
            for table in tables:
                table_body = table.find('tbody')
                if not table_body:
                    continue

                for row in table_body.find_all('tr'):
                    player_th = row.find('th', {'data-stat': 'player'})
                    if not player_th:
                        continue

                    player_name_tag = player_th.find('a')
                    if player_name_tag:
                        player_name = player_name_tag.text.strip()
                    else:
                        player_name = player_th.text.strip()

                    player_key = (player_name, team_name)
                    merged_player_stats[player_key]['match_date'] = date
                    merged_player_stats[player_key]['player'] = player_name
                    merged_player_stats[player_key]['team'] = team_name

                    for field in fields:
                        if field == 'player':
                            continue
                        cell = row.find('td', {'data-stat': field})
                        if cell:
                            merged_player_stats[player_key][field] = cell.text.strip()

    # Convert to a list of dictionaries
    merged_player_list = list(merged_player_stats.values())

    return merged_player_list





In [4]:
basic_match_data = fs.get_basic_match_data(schedule_url)[0]

response = fs.fetch_url_with_backoff(test_url)

soup = BeautifulSoup(response.text, 'html.parser')

match_details = fs.get_match_details(soup)

combined_match_details = {**basic_match_data, **match_details}
df_teams = pd.DataFrame([combined_match_details])


home_df = df_teams[[col for col in df_teams.columns if col != 'match_url']]
away_df = df_teams[[col for col in df_teams.columns if col != 'match_url']]

# Rename for merging
home_df = home_df.rename(columns={'home_team': 'team', 'away_team':'opp'})

away_df = away_df.rename(columns={'away_team': 'team','home_team':'opp'})

away_df = away_df.rename(columns=lambda col: col.replace('away_', 'team_') if col.startswith('away_') else (
                               col.replace('home_', 'opp_') if col.startswith('home_') else col))

home_df = home_df.rename(columns=lambda col: col.replace('home_', 'team_') if col.startswith('home_') else (
                               col.replace('away_', 'opp_') if col.startswith('away_') else col))



match_df = pd.concat([home_df, away_df])
match_df = match_df[['match_date','team','team_shots', 'team_shots_on_target','team_xG']]
# home_df
# combined_match_details


In [5]:
basic_match_data = fs.get_basic_match_data(schedule_url)

match_df




,match_date,team,team_shots,team_shots_on_target,team_xG
0,"Saturday August 24, 2024",Burnley,13,4,0.3
0,"Saturday August 24, 2024",Manchester City,1,1,1.9


In [ ]:
sot_data = get_sot_stats(test_url)

sot_df = pd.DataFrame(sot_data)
sot_df.columns

# Merge each side
total_sot_df = sot_df.merge(match_df, on=['match_date', 'team'], how='left')


total_sot_df.to_csv('../data/raw/sot_player_stats_TEST.csv')

# sot_data


In [ ]:
# # MAIN EXECUTION
# basic_match_data = fs.get_basic_match_data(schedule_url)

# for match in basic_match_data:

#     match_url = match['match_url']
#     # Match URL from 
#     match_details = fs.get_match_details(match_url)

### Match Context
Data Source: FBRef

In [ ]:
# Fields:
# expected possession share - moving average
# team implied goal probability - from market lines (proxy for attacking strength)
# opponent sot allowed per 90 - defensive weakness metric
# opponent xg allowed per 90 - xg conceded per 90
# game total goals line - market total (used to estimate match openness)


# Raw Data
# FBREF Data:
    # Team possession
    # SoT per 90
    # Opponent SoT Allowed per 90
    # xG per 90
    # Opponent xG Allowed per 90

# API Data:
    # Team Implied Goal Probability
    # Game total goals line

# what about opponent performance against player stats metrics per position?
#   i.e. moving average touches_att_3rd for LW over the past 5 games
